# test 마스크 직접 추론

6개 가중치로 예측합니다. YOLO 50% + DINO 25% + specialist 25%, Ni/Eu 경계 보정 −0.1. 저장된 `.npz` 확률은 읽지 않습니다.


## 1. 경로

`TEST_INPUT`은 test.zip 또는 원본 test 이미지 폴더입니다. 결과 폴더는 새 이름을 지정합니다.


In [ ]:
from pathlib import Path
from google.colab import drive
drive.mount('/content/drive')
PROJECT_ROOT = Path('/content/drive/MyDrive/KoMaP2026')
CODE_ROOT = Path('/content/drive/MyDrive/KoMaP2026_Submission_DirectInference_20261015')
TEST_INPUT = PROJECT_ROOT / 'test.zip'
OUTPUT_DIR = PROJECT_ROOT / 'submission_test_masks_01'
RUNS = PROJECT_ROOT / 'runs'
EXPECTED_COUNT = None
# runs 아래 폴더명: 재학습 모델을 쓰려면 여기서 변경합니다.
RUN_NAMES = {
    'yolo': 'yolo_eu_03_sizebalanced_20261011_seed2026',
    'dino': 'dinov3_hard_ablation_control_01',
    'primary': 'specialist_primary_fast_01',
    'ni': 'specialist_al3ni_fast_01',
    'eu': 'advanced_eu_balanced_ram_01',
    'al': 'specialist_al_fast_01',
}
CHECKPOINT_NAMES = {
    'yolo': 'best50.pt',
    'dino': 'best.pt', 'primary': 'best.pt',
    'ni': 'best.pt', 'eu': 'best.pt', 'al': 'best.pt',
}


## 2. 환경 확인

필수 패키지를 설치한 후, 추론 시작 전에 모든 의존성과 6개 모델의 로딩을 검사합니다. 설치 후 런타임 재시작 안내가 나오면 재시작하고 실행합니다.


In [ ]:
import os, sys, importlib, subprocess
assert (CODE_ROOT/'komap_direct_infer.py').is_file(), CODE_ROOT
os.environ['KOMAP_SUBMISSION_ROOT'] = str(CODE_ROOT)
sys.dont_write_bytecode = True
for name in list(sys.modules):
    if name.startswith('komap_'):
        del sys.modules[name]
sys.path[:] = [str(CODE_ROOT)] + [p for p in sys.path if p not in (str(CODE_ROOT), str(CODE_ROOT/'legacy_yolo_initial'))]
importlib.invalidate_caches()
subprocess.check_call([sys.executable, str(CODE_ROOT/'komap_submission_environment.py'), '--install'])


## 3. 가중치·추론 설정 검사

6개 best 파일과 5개 run의 export_index.json·연결된 manifest.json이 필요합니다. manifest는 클래스·TTA·가중치를 확인하는 용도이며 확률 .npz는 필요하지 않습니다.


In [ ]:
from komap_direct_infer import inspect_assets
RUN_PATHS = {role: RUNS / name for role, name in RUN_NAMES.items()}
ASSETS = inspect_assets(RUN_PATHS, checkpoint_names=CHECKPOINT_NAMES)
for role, asset in ASSETS.items():
    print(role, asset['checkpoint'], asset['checkpoint_sha256'][:12])


## 4. test 마스크 생성

CUDA GPU가 필요합니다. 원본 크기의 RGB 마스크와 test_masks.zip을 저장합니다. 정답 마스크는 사용하지 않습니다.


In [ ]:
from komap_direct_infer import predict_test
RESULT = predict_test(TEST_INPUT, OUTPUT_DIR, RUN_PATHS,
    expected_count=EXPECTED_COUNT, checkpoint_names=CHECKPOINT_NAMES)
print('마스크 폴더:', OUTPUT_DIR/'masks')
print('제출 마스크 ZIP:', RESULT['masks_zip'])


In [ ]:
from google.colab import files
files.download(RESULT['masks_zip'])
